# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation
* LangChain evaluation platform

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
# import getpass
# import os

# if not os.environ.get("GOOGLE_API_KEY"):
#   os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the lecture.

In [3]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

## Create our QandA application

In [ ]:
# We're going to import the retrieval QA chain. This will do retrieval over some documents. 
from langchain.chains import RetrievalQA

# We're going to import our favorite chat open AI language model.
from langchain.chat_models import ChatOpenAI
# from langchain.chat_models import init_chat_model

# We're going to import a document loader. This is going to be used to load some proprietary data that 
# we're going to combine with the language model. 
# In this case it's going to be in a CSV. So we're going to import the CSV loader.
from langchain.document_loaders import CSVLoader

# We're next going to import an index, the "VectorStoreIndexCreator". 
# This will help us create a vector store really easily.
from langchain.indexes import VectorstoreIndexCreator

# Finally, we're going to import a vector store. 
# There are many different types of vector stores which we covered. 
# This is really nice because it's an in-memory vector store and 
# It doesn't require connecting to an external database of any kind so it makes it really easy to get started.
from langchain.vectorstores import DocArrayInMemorySearch

In [5]:
# llm_model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [6]:
# load the same data that we used for the last lession. 
file = '../data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [7]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [8]:
from langchain.embeddings import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings()

# create the index with `DocArrayInMemorySearch` and `embedding_model`
# To create an index, we're going to specify two things. 
# First, we're going to specify the vector store class. 
# As mentioned before, we're going to use this vector store, as it's a particularly easy one to get started with. 
# After it's been created, we're then going to call "from_loaders", which takes in a list of document loaders. 
# We've only got one loader that we really care about, so that's what we're passing in here. 
# It's now been created and we can start to ask questions about it.

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embedding_model
).from_loaders([loader])

/var/folders/_9/kbclh8y12dz3_njd9xrldcm80000gp/T/ipykernel_68687/3148371321.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings()
/usr/local/genuin/code/personal/langchain/.venv/lib/python3.12/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [ ]:
# we are going to create a RetrievalQA chain by specifying 
# 1. Language model 
# 2. retriever 
# 3. verbosity 
# 4. Chain type = stuff 
llm = ChatOpenAI(temperature = 0.0, model=llm_model)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

/var/folders/_9/kbclh8y12dz3_njd9xrldcm80000gp/T/ipykernel_68687/1284373331.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature = 0.0, model=llm_model)


### Coming up with test datapoints

In [ ]:
# We need to figureout what are the data points 
# that we want to evaluate it on? 
# So there are few different methods for doing this. 
# 1st: come up with the data points that we think are good examples 
# In this method: 
# - find good examples 
# - come with example questions 
# - and ground truth answer to those questions 
# that will be used later to evaluate. 

# Cozy Comfort Pullover Set is the first example 
data[10]

Document(metadata={'source': '../data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [ ]:
# Stretch Down Hooded Jacket with bunch of details 
data[11]

Document(metadata={'source': '../data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

### Hard-coded examples

In [ ]:
# so from the details above we can create some examples by ourselves as follows. 
# it doesn't really scale that well. 
# have to figure out question answer pairs manually. 
# So is there a way we can automate this? 
# So one of the better ways to automate this is by using language model themselves. 
# Next we see QAGenerationChain -- which does exactly the same. 
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [ ]:
# QAGenerationChain 
# input: documents 
# output: QA pair from each document 
from langchain.evaluation.qa import QAGenerateChain

In [ ]:
# create an object 
# input: ChatOpenAI language model 
# obj. 
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))

In [ ]:
# the warning below can be safely ignored

In [ ]:
# we are using `example_gen_chain` with `apply_and_parse` method 
# to create `new_examples`

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

/var/folders/_9/kbclh8y12dz3_njd9xrldcm80000gp/T/ipykernel_68687/3125874183.py:1: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [ ]:
# check the results 
new_examples[0]

{'qa_pairs': {'query': "What materials are the Women's Campside Oxfords made of and what features contribute to their comfort?",
  'answer': "The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look. They also feature a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a moderate arch contour, EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern."}}

In [ ]:
# original document from which the question is generated 
data[0]

# so this works well to generate bunch of differene QA pairs 
# that way we can save time and improve quality. 

Document(metadata={'source': '../data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

### Combine examples

In [ ]:
# adding/appending the newly generated examples `new_examples` to 
# `examples` that we already created above. 
examples += new_examples

In [ ]:
# we generated the example but still we have not evaluated anything. 
# so to understand that, 
# let's take an example through the chain 
# and see the output it produces. 

# input: query
# output: answer 

# however, its slightly not so transprent 
# doesn't allow us to look into what is happening inside the chain? 

# what is the actual prompt going into the llm? 
# what are the documents that it retrives? 
# what if the chain was complex, it would have been difficult to do any analysis. 

# we not just want the final answer but also want to be transperant 
# for that we have next, our langchain debug. 
qa.run(examples[0]["query"])

/var/folders/_9/kbclh8y12dz3_njd9xrldcm80000gp/T/ipykernel_68687/1223946598.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  qa.run(examples[0]["query"])




> Entering new RetrievalQA chain...

> Finished chain.


'Yes, the Cozy Comfort Pullover Set does have side pockets.'

## Manual Evaluation

In [ ]:
# set langchain debug = True 
import langchain
langchain.debug = True

In [ ]:
# rerun the same example as above 

# often times, when getting the wrong answer -> its not always LLM that is massing up. 
# its the retriver that might be getting the wrong context. 
# so this process helps debug what's going on under the hoods. 
qa.run(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": ": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-

'Yes, the Cozy Comfort Pullover Set does have side pockets.'

In [22]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [ ]:
# we looked at the single Q&A debug. 
# What about all the examples that we created? 
# How are we going to debug them? 
# one way could be: manually running all the examples through the chain. 
# and look at the outputs. 
# and try to figureout whats going on? correct, incorrect, partially correct? 
# So to achieve scalability 
# can we ask the LLM to do it? 

# let's create all the predictions for all of our examples. 
modified_examples = [example['qa_pairs'] for example in examples[2:]]

predictions = qa.apply(modified_examples)

/var/folders/_9/kbclh8y12dz3_njd9xrldcm80000gp/T/ipykernel_68687/2389102651.py:3: LangChainDeprecationWarning: The method `Chain.apply` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~batch` instead.
  predictions = qa.apply(modified_examples)




> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [ ]:
# now that we have got these examples 
# we can think about evaluating them 
# to do it: import QAEvalChain. 
from langchain.evaluation.qa import QAEvalChain

In [ ]:
# create a chain using `QAEvalChain` using a language model
# why language model? 
# because we are using llm as a judge here. 
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = QAEvalChain.from_llm(llm)

In [26]:
examples = modified_examples
graded_outputs = eval_chain.evaluate(examples, predictions)

In [29]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()

Example 0:
Question: What materials are the Women's Campside Oxfords made of and what features contribute to their comfort?
Real Answer: The Women's Campside Oxfords are made of soft canvas material for a broken-in feel and look. They also feature a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, a moderate arch contour, EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern.
Predicted Answer: The Women's Campside Oxfords are made of soft canvas material, and they feature a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. The EVA foam midsole provides cushioning and support, while the chain-tread-inspired molded rubber outsole with a modified chain-tread pattern adds to the comfort and support of the shoes.
Predicted Grade: CORRECT

Example 1:
Question: What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?
Real 

In [28]:
graded_outputs[0]

{'results': 'CORRECT'}